# 10 - Audio File Demo (software-only, no hardware)

**Objective:** Pick one real example audio clip per class (gunshot / chainsaw / background) from the dataset, run it through the final INT8 TFLite model exactly as an embedded device would (feature extraction -> quantize -> invoke -> dequantize -> argmax), and print the predicted class + confidence.

In [1]:
import sys, os
from pathlib import Path
ML_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ML_ROOT / 'preprocessing'))
sys.path.insert(0, str(ML_ROOT / 'scripts'))
sys.path.insert(0, str(ML_ROOT / 'models'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
FIG_DIR = ML_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('ML_ROOT =', ML_ROOT)


ML_ROOT = /Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml


In [2]:
import soundfile as sf, tensorflow as tf, json
from feature_extraction import get_feature_extractor
extractor = get_feature_extractor()
final_meta = json.load(open(ML_ROOT/'models'/'final'/'final_metadata.json'))
mean, std = final_meta['feature_mean'], final_meta['feature_std']
CLASSES = ['background', 'chainsaw', 'gunshot']
interp = tf.lite.Interpreter(model_path=str(ML_ROOT/'models'/'final'/'forest_acoustic_int8.tflite'))
interp.allocate_tensors()
in_d = interp.get_input_details()[0]; out_d = interp.get_output_details()[0]
in_scale, in_zero = in_d['quantization']
out_scale, out_zero = out_d['quantization']

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [3]:
master = pd.read_csv(ML_ROOT/'datasets'/'v1'/'test_v1.csv', low_memory=False)
examples = {cls: ML_ROOT / master[master['class']==cls].iloc[0]['processed_path'] for cls in CLASSES}
examples

{'background': PosixPath('/Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml/datasets/processed/fsc22_2_10206_0000.wav'),
 'chainsaw': PosixPath('/Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml/datasets/processed/fsc22_11_11101_0000.wav'),
 'gunshot': PosixPath('/Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml/datasets/processed/c3gd_13-nj_farm-savage_mh110-DAVE-3fPPqTzCQ_WZ-8_0000.wav')}

## Run inference on each example clip

In [4]:
def predict(wav_path):
    audio, sr = sf.read(wav_path)
    mel = extractor.extract_mel_spectrogram(audio, sr)
    T = mel.shape[1]
    x = ((mel - mean) / std)[np.newaxis, ..., np.newaxis].astype(np.float32)
    xq = np.round(x / in_scale + in_zero).astype(np.int8)
    interp.set_tensor(in_d['index'], xq)
    interp.invoke()
    out_q = interp.get_tensor(out_d['index'])[0]
    probs = (out_q.astype(np.float32) - out_zero) * out_scale
    probs = np.exp(probs) / np.exp(probs).sum()  # renormalize as a proxy confidence
    pred_idx = int(np.argmax(out_q))
    return CLASSES[pred_idx], float(probs[pred_idx])

for cls, path in examples.items():
    pred_class, conf = predict(path)
    print(f'True: {cls:12s} -> Predicted: {pred_class:12s} (confidence proxy: {conf:.3f})  [{path.name}]')

True: background   -> Predicted: background   (confidence proxy: 0.507)  [fsc22_2_10206_0000.wav]
True: chainsaw     -> Predicted: chainsaw     (confidence proxy: 0.412)  [fsc22_11_11101_0000.wav]
True: gunshot      -> Predicted: gunshot      (confidence proxy: 0.573)  [c3gd_13-nj_farm-savage_mh110-DAVE-3fPPqTzCQ_WZ-8_0000.wav]


## Conclusion

This is a pure software demonstration of the end-to-end inference path (feature extraction -> INT8 quantize -> TFLite invoke -> class prediction) that would run on-device. No physical ESP32-S3/INMP441 hardware was used or available — real-time I2S capture and on-device execution are explicitly listed as blocked in `reports/FINAL_MODEL_REPORT.md`.